<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/TEXTUREALCHEMY_PONY_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TextureAlchemy + Pony V6 XL — A100

Opens **from GitHub**. **Does not use your local GPU.**

| | |
|---|---|
| GPU | Runtime → **A100** + High-RAM |
| Code | `amtarr/ComfyUI-TextureAlchemy` cloned from GitHub |
| Weights | Drive `MyDrive/models/ponyDiffusionV6XL_v6StartWithThisOne.safetensors` |
| IN | Drive `MyDrive/CANINE_V300/` — BodyMatt / EyeMatt cards |
| OUT | Drive `MyDrive/CANINE_V300/outputs/` |

This is **not Hunyuan**. UVs stay yours. Pony paints tiles; TextureAlchemy makes them seamless.

Pony is a character checkpoint. Tiles will look like painted fur, not a photo albedo. Inpaint/ControlNet is a later pass.

If the Pony file is missing on Drive, this notebook **stops**. It will not silently swap gay621.


In [ ]:
# 0) A100 gate
import torch, sys
print("python", sys.version.split()[0])
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram:.1f} GB")
if vram < 24:
    raise SystemExit(
        f"This notebook wants A100 (>=24GB). This box is {vram:.1f}GB ({name}). "
        "Runtime → Change runtime type → A100. Do not run on T4."
    )
print("A100-class OK.")


In [ ]:
# 1) Drive = files. Code stays on GitHub.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
MODELS = DRIVE / "models"
IN_DIR = DRIVE / "CANINE_V300"
OUT_DIR = IN_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PONY = MODELS / "ponyDiffusionV6XL_v6StartWithThisOne.safetensors"

print("IN ", IN_DIR, "exists", IN_DIR.exists())
if IN_DIR.exists():
    print("   ", sorted(p.name for p in IN_DIR.iterdir())[:20])
print("PONY", PONY, "exists", PONY.exists(), "bytes", PONY.stat().st_size if PONY.exists() else 0)
if not PONY.exists():
    raise SystemExit(
        "Put ponyDiffusionV6XL_v6StartWithThisOne.safetensors in MyDrive/models/ "
        "next to gay621, then re-run this cell. Will not fall back to another ckpt."
    )


In [ ]:
# 2) Clone TextureAlchemy from GitHub (source of truth)
import subprocess
from pathlib import Path

dst = Path("/content/ComfyUI-TextureAlchemy")
if not dst.exists():
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/amtarr/ComfyUI-TextureAlchemy.git", str(dst)],
        check=False,
    )
    if r.returncode != 0:
        raise SystemExit(f"git clone TextureAlchemy failed: {r.returncode}")
sha = subprocess.check_output(["git", "-C", str(dst), "rev-parse", "--short", "HEAD"], text=True).strip()
print("TextureAlchemy", sha)


In [ ]:
# 3) Diffusers + Pony. Colab torch stays. Pony is standard SDXL.
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "omegaconf"],
    check=False,
)
if r.returncode != 0:
    raise SystemExit(f"pip failed: {r.returncode}")
print("pip ok")


In [ ]:
# 4) Load Pony from Drive (not from this runtime disk as the source of truth)
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
from pathlib import Path

PONY = Path("/content/drive/MyDrive/models/ponyDiffusionV6XL_v6StartWithThisOne.safetensors")
pipe = StableDiffusionXLPipeline.from_single_file(
    str(PONY), torch_dtype=torch.float16, use_safetensors=True
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
print("loaded Pony V6 XL on", torch.cuda.get_device_name(0))


In [ ]:
# 5) Collect BodyMatt + EyeMatt cards from Drive
from pathlib import Path

IN_DIR = Path("/content/drive/MyDrive/CANINE_V300")
cards = []
for sub in ("BodyMatt", "EyeMatt"):
    d = IN_DIR / sub
    if not d.exists():
        print("MISSING", d)
        continue
    for p in sorted(d.glob("*.png")):
        cards.append(p)
print("cards", len(cards))
for p in cards[:8]:
    print(" ", p.relative_to(IN_DIR))
if not cards:
    raise SystemExit("No BodyMatt/EyeMatt PNGs in MyDrive/CANINE_V300/. Upload the mats folders.")


In [ ]:
# 6) Paint tiles with Pony, seamless via TextureAlchemy. Writes Drive.
import sys, time
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms.functional import to_tensor, to_pil_image

sys.path.insert(0, "/content/ComfyUI-TextureAlchemy")
from texture_utils import SeamlessTiling

IN_DIR = Path("/content/drive/MyDrive/CANINE_V300")
OUT_DIR = IN_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POS = (
    "score_9, score_8_up, score_7_up, source_furry, rating_safe, "
    "anthro canine short fur texture tile, cream tan fur, albedo, "
    "flat lighting, no face, no eyes, no clothes, seamless"
)
NEG = "score_4, score_5, human, photoreal, text, watermark, clothing, extra limbs, scenery"

# A100 can do more; keep first pass small so a missing folder fails cheap.
MAX_CARDS = 8
SEED = 42
STEPS = 24
seamer = SeamlessTiling()

def pil_to_bhwc(im):
    t = to_tensor(im.convert("RGB")).unsqueeze(0).permute(0, 2, 3, 1).to("cuda")
    return t

def bhwc_to_pil(t):
    x = t.detach().float().cpu()[0].permute(2, 0, 1).clamp(0, 1)
    return to_pil_image(x)

n = 0
for i, path in enumerate(cards[:MAX_CARDS]):
    tag = path.stem
    w, h = Image.open(path).size
    tw = max(64, min(768, (w + 63) // 64 * 64))
    th = max(64, min(768, (h + 63) // 64 * 64))
    print(f"[{i}] {tag} {tw}x{th}", flush=True)
    gen = pipe(
        prompt=f"{POS}, {tag}",
        negative_prompt=NEG,
        width=tw,
        height=th,
        num_inference_steps=STEPS,
        guidance_scale=7.0,
        generator=torch.Generator("cuda").manual_seed(SEED + i),
    ).images[0]
    raw_path = OUT_DIR / f"{tag}_pony.png"
    gen.save(raw_path)
    tiled, _mask = seamer.make_seamless(pil_to_bhwc(gen), method="blend_edges", blend_width=0.1)
    seam = bhwc_to_pil(tiled)
    seam_path = OUT_DIR / f"{tag}_seamless.png"
    seam.save(seam_path)
    print("  wrote", raw_path.name, seam_path.name, flush=True)
    n += 1
    if i % 2 == 1:
        torch.cuda.empty_cache()

print("DONE", n, "tiles →", OUT_DIR)
print("raise MAX_CARDS and re-run this cell for the rest.")
